# E3C – Semantic–Collaborative Agreement Analysis (Google Colab)

Notebook này chạy script E3C trên Google Colab. **Không cần cài Python trên máy.**

**Đầu vào bắt buộc:** `users.csv` (semantic/collaborative embeddings cùng không gian biểu diễn) và `metrics.csv` (Recall@20, NDCG@20 theo user, model, seed). Các file kết quả tổng hợp E3A/E3B **không đủ** để thực hiện E3C. Notebook không tự trích xuất embeddings/checkpoint từ repository RLMRec.

In [ ]:
!pip -q install numpy pandas

## 1. Tải code và dữ liệu lên Colab
Chạy ô sau, chọn `E3C_agreement_analysis.py`, `users.csv`, `metrics.csv` và tùy chọn `items.csv`. Có thể chọn nhiều file cùng lúc.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Đã tải lên:', list(uploaded))

## 2. Chạy phân tích
Giữ nguyên tên file và nhãn mô hình nếu đúng với dữ liệu CSV. Nếu file chưa đủ hoặc embeddings không hợp lệ, script sẽ báo lỗi thay vì tự tạo số liệu.

In [ ]:
import os, subprocess, sys
required = ['E3C_agreement_analysis.py', 'users.csv', 'metrics.csv']
missing = [x for x in required if not os.path.isfile(x)]
if missing:
    raise FileNotFoundError('Thiếu file: ' + ', '.join(missing))
cmd = [sys.executable, 'E3C_agreement_analysis.py', '--users', 'users.csv', '--metrics', 'metrics.csv', '--baseline', 'LightGCN', '--treatment', 'RLMRec-Con', '--output', 'e3c_results']
if os.path.isfile('items.csv'):
    cmd.extend(['--items', 'items.csv'])
print('Đang chạy E3C...')
subprocess.run(cmd, check=True)

## 3. Xem và tải kết quả

In [ ]:
import pandas as pd
from IPython.display import display
from pathlib import Path
out = Path('e3c_results')
for f in sorted(out.glob('*.csv')):
    print('\n', f.name)
    display(pd.read_csv(f).head(12))
import shutil
shutil.make_archive('E3C_results', 'zip', out)
from google.colab import files
files.download('E3C_results.zip')

## Lưu ý phương pháp
Cosine similarity chỉ có ý nghĩa khi hai loại embeddings ở **cùng không gian tọa độ**. Không dùng trực tiếp semantic embeddings thô của LLM và graph embeddings nếu chưa có phép biến đổi phù hợp. Nên dùng biểu diễn trước alignment để tránh kết luận vòng tròn. Các nhóm Low/Medium/High trong notebook được phân chia theo phân vị trên người dùng.